# 05: GLMM and Machine Learning

This notebook implements Generalized Linear Mixed Models (GLMM) and machine learning models (Random Forest and XGBoost).

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from loguru import logger

from src import io, effects, eval as eval_module

# Fix matplotlib overflow error with large datasets
plt.rcParams['agg.path.chunksize'] = 10000

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style('whitegrid')

## 1. Load Data

In [ ]:
df = io.load_raw_data(Path('../data/processed/cosinor_data.parquet'))
logger.info(f"Loaded data with shape: {df.shape}")
df.head()

## 2. Prepare Data for ML

In [ ]:
# Select features (use rolling stats and time features if available)
feature_cols = [col for col in df.columns if any([
    'rolling' in col,
    'lag' in col,
    col in ['hour', 'day_of_week', 'is_weekend', 'hmm_state']
])]

# Fallback to basic features if advanced ones don't exist
if not feature_cols:
    if 'hour' in df.columns:
        feature_cols = ['hour']
    else:
        # Create a simple feature
        df['hour'] = pd.to_datetime(df['timestamp']).dt.hour
        feature_cols = ['hour']

target_col = 'activity'

print(f"\nUsing {len(feature_cols)} features:")
print(feature_cols[:10])  # Show first 10
print(f"\nTarget: {target_col}")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = effects.prepare_ml_data(
    df,
    target_column=target_col,
    feature_columns=feature_cols,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 3. Random Forest Model

In [ ]:
# Fit Random Forest
rf_results = effects.fit_random_forest(
    X_train, y_train, X_test, y_test,
    task='regression',
    n_estimators=100,
    max_depth=10,
    random_state=RANDOM_STATE
)

print("\nRandom Forest Results:")
print(f"  Train R²: {rf_results['train_score']:.4f}")
print(f"  Test R²: {rf_results['test_score']:.4f}")

In [ ]:
# Evaluate RF predictions
rf_model = rf_results['model']
y_pred_rf = rf_model.predict(X_test)

rf_metrics = eval_module.evaluate_regression(y_test, y_pred_rf)
print("\nRandom Forest Metrics:")
for metric, value in rf_metrics.items():
    print(f"  {metric}: {value:.4f}")

## 4. XGBoost Model

In [ ]:
# Fit XGBoost
xgb_results = effects.fit_xgboost(
    X_train, y_train, X_test, y_test,
    task='regression',
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE
)

print("\nXGBoost Results:")
print(f"  Train R²: {xgb_results['train_score']:.4f}")
print(f"  Test R²: {xgb_results['test_score']:.4f}")

In [ ]:
# Evaluate XGB predictions
xgb_model = xgb_results['model']
y_pred_xgb = xgb_model.predict(X_test)

xgb_metrics = eval_module.evaluate_regression(y_test, y_pred_xgb)
print("\nXGBoost Metrics:")
for metric, value in xgb_metrics.items():
    print(f"  {metric}: {value:.4f}")

## 5. Model Comparison

In [ ]:
# Compare models
comparison_df = effects.compare_models(
    X_train, y_train, X_test, y_test,
    task='regression',
    random_state=RANDOM_STATE
)

print("\nModel Comparison:")
print(comparison_df)

## 6. Feature Importance

In [ ]:
# Plot feature importance for Random Forest
eval_module.plot_feature_importance(
    feature_cols,
    rf_results['feature_importances'],
    top_n=min(20, len(feature_cols)),
    save_path='../outputs/rf_feature_importance.png'
)
logger.info("Feature importance plot saved")

## 7. Prediction Visualization

In [ ]:
# Plot predictions vs actual
eval_module.plot_predictions(
    y_test,
    y_pred_xgb,
    title='XGBoost: Predictions vs Actual',
    save_path='../outputs/xgb_predictions.png'
)

# Plot residuals
eval_module.plot_residuals(
    y_test,
    y_pred_xgb,
    save_path='../outputs/xgb_residuals.png'
)

logger.info("Prediction plots saved")

## 8. GLMM (Optional)

Fit a Generalized Linear Mixed Model if pymer4 is available.

In [ ]:
# Prepare data for GLMM
if 'subject_id' in df.columns:
    df_glmm = df.dropna(subset=feature_cols + [target_col, 'subject_id'])
    
    # Fit GLMM
    glmm_results = effects.fit_glmm(
        df_glmm,
        formula=f"{target_col} ~ hour",
        random_effects="(1|subject_id)",
        family='gaussian'
    )
    
    if 'error' not in glmm_results:
        print("\nGLMM Results:")
        print(glmm_results['fitted'])
    else:
        print(f"\nGLMM: {glmm_results['error']}")
else:
    print("\nGLMM: subject_id not available, skipping mixed effects model")

## 9. Save Final Results

In [ ]:
# Create summary dataframe
results_summary = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost'],
    'Train_R2': [rf_results['train_score'], xgb_results['train_score']],
    'Test_R2': [rf_results['test_score'], xgb_results['test_score']],
    'RMSE': [rf_metrics['rmse'], xgb_metrics['rmse']],
    'MAE': [rf_metrics['mae'], xgb_metrics['mae']],
})

output_path = Path('../outputs/model_results.csv')
results_summary.to_csv(output_path, index=False)
logger.success(f"Results summary saved to {output_path}")

print("\nFinal Results Summary:")
print(results_summary)

## 10. Log Parameters

In [ ]:
params = {
    'random_state': RANDOM_STATE,
    'n_features': len(feature_cols),
    'test_size': 0.2,
    'rf_n_estimators': 100,
    'rf_max_depth': 10,
    'xgb_n_estimators': 100,
    'xgb_learning_rate': 0.1,
    'xgb_max_depth': 6,
}
logger.info(f"ML parameters: {params}")

print("\nPipeline Parameters:")
for key, value in params.items():
    print(f"  {key}: {value}")